# Round 4: closing the gap - S6E10

The OOF checkpoint (round 3) put the incumbent raw LightGBM at 0.95847 mean OOF AUC and settled two
questions: the engineered feature set is dropped for tree models (all 5 paired fold deltas negative),
and blending showed no signal. The local target is 0.96 - remaining gap about +0.0015.

This round: three informed single-knob moves under the IDENTICAL 5-fold OOF protocol (same folds,
seed 42), so every new number pairs exactly with the round-3 incumbent folds. Two random-forest moves
(Andrea's preferred family) and one LightGBM move:

- **Exp 12: RF min_samples_leaf=10** - leaf smoothing was the one RF change that produced a real gain
  (Exp 1 -> Exp 4); this is one more step along that same direction.
- **Exp 13: RF max_depth=25** (leaf back to 5) - depth limiting is the other standard RF regularizer;
  untested so far.
- **Exp 14: LightGBM learning_rate=0.03** (n_estimators raised to 3000, early stop 50) - the standard
  reliable small-gain GBM move; round-3 best_iterations were 183-555, so headroom exists.

Reading rules: same as round 3 - a mean paired delta that clears 2x the spread of the per-fold deltas
is a WORKING SIGNAL, not proof; anything smaller is NO SIGNAL. Metrics are reported BOTH ways
(mean of fold AUCs and pooled AUC over all OOF predictions) so cross-experiment comparisons use one
consistent basis. A candidate only earns a submission file if it beats the incumbent's OOF mean with a
working signal - and even then it stays Andrea's call to submit.


In [1]:
import inspect
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

# Round-3 incumbent (Exp 8) per-fold AUCs on these exact folds - the paired reference.
INCUMBENT_FOLDS = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]
INCUMBENT_OOF_MEAN = float(np.mean(INCUMBENT_FOLDS))  # 0.95847


In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file():
    if Path('playground-series-s6e10/data/train.csv').is_file():
        competition_dir = Path('playground-series-s6e10')
    elif Path('../data/train.csv').is_file():
        competition_dir = Path('..')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])

def to_lgbm(df):
    # LightGBM-native copy: categoricals as 'category', NaN left alone. sklearn frames keep object dtype.
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def rf_pipeline(train_frame, min_samples_leaf=5, max_depth=None, n_estimators=400):
    numeric_columns = train_frame.columns.difference(CATEGORICAL).tolist()
    preprocessor = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_columns),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CATEGORICAL),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
            max_depth=max_depth, n_jobs=-1, random_state=RANDOM_STATE)),
    ])

def oof_lgbm(feat_frame, label, name, learning_rate=0.05, num_leaves=63, n_estimators=1000):
    frame = to_lgbm(feat_frame)
    oof = np.zeros(len(frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(frame, label), 1):
        X_tr, y_tr = frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = frame.iloc[va_idx], label.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        model = lgb.LGBMClassifier(learning_rate=learning_rate, num_leaves=num_leaves,
                                   n_estimators=n_estimators, n_jobs=-1,
                                   random_state=RANDOM_STATE, verbose=-1)
        # LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
        # use whichever signature the installed version supports (no warning either way).
        if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
            model.fit(X_fit, y_fit, eval_X=X_es, eval_y=y_es, eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        else:
            model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        p = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = p
        a = roc_auc_score(y_va, p)
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def oof_rf(feat_frame, label, name, **rf_kwargs):
    oof = np.zeros(len(feat_frame))
    aucs = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        rf = rf_pipeline(X_tr, **rf_kwargs)
        rf.fit(X_tr, y_tr)
        p = rf.predict_proba(X_va)[:, 1]
        oof[va_idx] = p
        a = roc_auc_score(y_va, p)
        aucs.append(a)
        print(f'  {name} fold {fold}: AUC {a:.5f}')
    return oof, aucs

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features (no engineering this round)')


Full train: 699,635 rows, 21 raw features (no engineering this round)


## Exp 12: random forest, min_samples_leaf=10

One knob from the Exp 4/10 config (400 trees, leaf=5): leaf=10. Hypothesis: the leaf-smoothing
direction is the only RF change that produced a real gain so far, so one more step along it is the
most informed RF move available.


In [3]:
print('Exp 12: RF 400 trees, min_samples_leaf=10')
oof12, aucs12 = oof_rf(features, target, 'E12', min_samples_leaf=10)
print(f'Exp 12 mean OOF AUC: {np.mean(aucs12):.5f} +/- {np.std(aucs12):.5f} | pooled: {roc_auc_score(target, oof12):.5f}')


Exp 12: RF 400 trees, min_samples_leaf=10


  E12 fold 1: AUC 0.95697


  E12 fold 2: AUC 0.95601


  E12 fold 3: AUC 0.95726


  E12 fold 4: AUC 0.95710


  E12 fold 5: AUC 0.95702
Exp 12 mean OOF AUC: 0.95687 +/- 0.00044 | pooled: 0.95687


## Exp 13: random forest, max_depth=25

One knob from the Exp 4/10 config (400 trees, leaf=5): cap depth at 25. Hypothesis: with 700k rows
the trees grow very deep; a depth cap is the other standard regularizer and is untested so far.


In [4]:
print('Exp 13: RF 400 trees, leaf=5, max_depth=25')
oof13, aucs13 = oof_rf(features, target, 'E13', min_samples_leaf=5, max_depth=25)
print(f'Exp 13 mean OOF AUC: {np.mean(aucs13):.5f} +/- {np.std(aucs13):.5f} | pooled: {roc_auc_score(target, oof13):.5f}')


Exp 13: RF 400 trees, leaf=5, max_depth=25


  E13 fold 1: AUC 0.95712


  E13 fold 2: AUC 0.95625


  E13 fold 3: AUC 0.95741


  E13 fold 4: AUC 0.95722


  E13 fold 5: AUC 0.95715
Exp 13 mean OOF AUC: 0.95703 +/- 0.00040 | pooled: 0.95703


## Exp 14: LightGBM learning_rate=0.03

One knob from the incumbent (Exp 8): lr 0.05 -> 0.03, with n_estimators raised to 3000 so early
stopping (50 rounds, inner 10% split) still decides tree count. Hypothesis: slower learning with more
trees is the standard small-gain GBM move, and round-3 best_iterations (183-555) leave headroom.


In [5]:
print('Exp 14: LightGBM lr=0.03, num_leaves=63, early stop 50')
oof14, aucs14, iters14 = oof_lgbm(features, target, 'E14', learning_rate=0.03, n_estimators=3000)
print(f'Exp 14 mean OOF AUC: {np.mean(aucs14):.5f} +/- {np.std(aucs14):.5f} | pooled: {roc_auc_score(target, oof14):.5f}')


Exp 14: LightGBM lr=0.03, num_leaves=63, early stop 50


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E14 fold 1: AUC 0.95843 (best_iteration=375)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E14 fold 2: AUC 0.95750 (best_iteration=463)


...

  E14 fold 3: AUC 0.95916 (best_iteration=409)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E14 fold 4: AUC 0.95842 (best_iteration=329)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E14 fold 5: AUC 0.95896 (best_iteration=468)
Exp 14 mean OOF AUC: 0.95849 +/- 0.00058 | pooled: 0.95848


## Results, verdicts, and refit guard

Paired per-fold deltas against the round-3 incumbent folds (same folds, seed 42). Labels: WORKING
SIGNAL when the mean paired delta clears 2x the spread of the per-fold deltas (descriptive, not
proof), NO SIGNAL otherwise. The refit guard compares against the standing incumbent - a repeat of
round 2's within-round-only mistake is structurally impossible here.


In [6]:
def paired_verdict(new_aucs, ref_aucs=INCUMBENT_FOLDS):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return deltas, mean_d, spread, label

results = {}
for exp_id, name, aucs, oof in [
    ('12', 'RF leaf=10', aucs12, oof12),
    ('13', 'RF max_depth=25', aucs13, oof13),
    ('14', 'LGBM lr=0.03', aucs14, oof14),
]:
    deltas, mean_d, spread, label = paired_verdict(aucs)
    results[exp_id] = dict(name=name, aucs=aucs, oof=oof, mean_d=mean_d, label=label)
    print(f'Exp {exp_id} {name}: mean {np.mean(aucs):.5f} +/- {np.std(aucs):.5f} | pooled {roc_auc_score(target, oof):.5f}')
    print(f'  paired deltas vs incumbent folds: ' + ' '.join(f'{d:+.5f}' for d in deltas) + f' (mean {mean_d:+.5f})')
    print(f'  {label}')
    print()

best_oof = max([INCUMBENT_OOF_MEAN] + [float(np.mean(r['aucs'])) for r in results.values()])
print(f'Best OOF mean incl. incumbent: {best_oof:.5f} - >=0.96: {best_oof >= 0.96} (descriptive; target claims stay cautious - repeated selection on this data)')

winners = [k for k, r in results.items()
           if r['mean_d'] > 0 and 'WORKING SIGNAL: better' in r['label']]
print()
if not winners:
    print('No candidate earned a refit: incumbent Exp 2 raw LightGBM stands, submission_lgbm.csv remains the best submission candidate.')
else:
    win = max(winners, key=lambda k: np.mean(results[k]['aucs']))
    r = results[win]
    print(f'Winner: Exp {win} ({r["name"]}) - refits on all rows below, logged as a working signal, exploratory.')


Exp 12 RF leaf=10: mean 0.95687 +/- 0.00044 | pooled 0.95687
  paired deltas vs incumbent folds: -0.00154 -0.00150 -0.00181 -0.00131 -0.00183 (mean -0.00160)
  WORKING SIGNAL: worse (descriptive, not proof)



Exp 13 RF max_depth=25: mean 0.95703 +/- 0.00040 | pooled 0.95703
  paired deltas vs incumbent folds: -0.00139 -0.00126 -0.00166 -0.00119 -0.00170 (mean -0.00144)
  WORKING SIGNAL: worse (descriptive, not proof)

Exp 14 LGBM lr=0.03: mean 0.95849 +/- 0.00058 | pooled 0.95848
  paired deltas vs incumbent folds: -0.00008 -0.00001 +0.00009 +0.00001 +0.00011 (mean +0.00002)
  NO SIGNAL (within 2x per-fold delta spread - no action)

Best OOF mean incl. incumbent: 0.95849 - >=0.96: False (descriptive; target claims stay cautious - repeated selection on this data)

No candidate earned a refit: incumbent Exp 2 raw LightGBM stands, submission_lgbm.csv remains the best submission candidate.


In [7]:
# Refit ONLY fires when the summary above found a working-signal winner over the standing incumbent.
if not winners:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    if win in ('12', '13'):
        kwargs = {'min_samples_leaf': 10} if win == '12' else {'min_samples_leaf': 5, 'max_depth': 25}
        final = rf_pipeline(features, **kwargs)
        final.fit(features, target)
        test_probabilities = final.predict_proba(test_features)[:, 1]
    else:
        final = lgb.LGBMClassifier(learning_rate=0.03, num_leaves=63,
                                   n_estimators=int(np.mean(iters14)), n_jobs=-1,
                                   random_state=RANDOM_STATE, verbose=-1)
        final.fit(to_lgbm(features), target)
        test_probabilities = final.predict_proba(to_lgbm(test_features))[:, 1]
    submission = pd.read_csv(data_dir / 'sample_submission.csv')
    submission['satisfaction'] = test_probabilities
    assert len(submission) == len(test)
    assert submission['id'].equals(test['id'])
    out_path = competition_dir / 'submission_round4.csv'
    submission.to_csv(out_path, index=False)
    print(f"Saved {len(submission):,} predictions to {out_path} - submitting stays Andrea's call.")


No submission file written this round.


## What happens next

Round 4 produced no working signal, so the incumbent (Exp 2 raw LightGBM) stands. Round 5 moves,
none gated on a new winner:

- num_leaves one step up (63 -> 127) directly on the incumbent under the same protocol.
- Categorical representation on both families: ordinal encoding for the random forest, one-hot for
  LightGBM - the representation itself is an untested lever.
- Error-analysis-driven features: segment the incumbent's OOF predictions, find where ranking is
  weakest, and test interaction terms motivated by that analysis (a NEW hypothesis - distinct from
  the dropped round-2/3 feature set).
- An OOF cache (see the executor addendum below) now persists predictions, so pooled and paired
  checks no longer need reruns.


## Executor addendum: OOF cache (disclosed)

Added by the executor per the round-4 brief: persist this round's OOF predictions, fold AUCs, run configs, and dataset identity to the gitignored `oof_cache/` directory so later rounds can run pooled/paired checks without rerunning. No experiment code changed; no row-level data enters git.

In [8]:
import hashlib, json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
train_path = competition_dir / 'data' / 'train.csv'
h = hashlib.sha256()
with open(train_path, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round4_oof.npz',
         oof12=np.asarray(oof12), aucs12=np.asarray(aucs12),
         oof13=np.asarray(oof13), aucs13=np.asarray(aucs13),
         oof14=np.asarray(oof14), aucs14=np.asarray(aucs14),
         iters14=np.asarray(iters14),
         incumbent_folds=np.asarray(INCUMBENT_FOLDS),
         y=np.asarray(target).astype(int))
meta = {
    'round': 4,
    'protocol': '5-fold stratified OOF, seed 42; LightGBM inner-10% early stop per fold',
    'dataset': {'file': 'data/train.csv', 'rows': int(len(target)), 'sha256': h.hexdigest()},
    'configs': {
        'exp12': 'RandomForest 400 trees, min_samples_leaf=10',
        'exp13': 'RandomForest 400 trees, min_samples_leaf=5, max_depth=25',
        'exp14': 'LightGBM lr=0.03, num_leaves=63, n_estimators=3000, early stop 50',
    },
    'incumbent_folds_source': 'round 3 measured folds: 0.95851 0.95751 0.95907 0.95841 0.95885',
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
(cache_dir / 'round4_oof.json').write_text(json.dumps(meta, indent=2))
print('OOF cache written to', cache_dir)


OOF cache written to oof_cache
